# Part 2 - PCA (variant 1)

In [ ]:
import os
import os.path as op
import os, sys, time
sys.path.append('.')

from paths import *
from tqdm.notebook import trange, tqdm
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn.image import iter_img, resample_to_img
from nilearn.plotting import plot_stat_map, show
from sklearn.decomposition import FastICA, PCA
from nilearn.masking import compute_epi_mask

from nilearn import datasets
from scipy.stats import zscore

save_results = 'results/'
if not os.path.exists(save_results):
    os.makedirs(save_results)
    
%matplotlib inline

## 0. Load the preprocessed fMRI data

In [ ]:
img = nib.load(SMOOTH_PATH + ".nii.gz")
affine = img.affine
data = np.asanyarray(img.dataobj)                 
tr = float(img.header.get_zooms()[3])
print(data.shape, data.dtype, "| TR =", round(tr, 3), "s")

## 1. Data matrix: brain mask, one timepoint = one sample

In [ ]:
vol_shape = data.shape[:-1]
n_vols = data.shape[-1]
print("volume shape:", vol_shape, "| number of volumes:", n_vols)

In [ ]:
mean_vol = data.mean(axis=3)
slice_non_background = compute_epi_mask(nib.Nifti1Image(mean_vol, affine)).get_fdata() > 0
print("brain voxels:", int(slice_non_background.sum()), "of", int(np.prod(vol_shape)))

samples = np.ascontiguousarray(data[slice_non_background].T)      
del data, mean_vol                                                
print("samples:", samples.shape)

In [ ]:
ZSCORE_VOXELS = False    

spatial_means = samples.mean(axis=1)               
row_means = spatial_means[:, None]                 
X = samples - row_means
del samples

if ZSCORE_VOXELS:
    sd = X.std(axis=0)
    sd[sd == 0] = 1
    X = (X - X.mean(axis=0)) / sd

print("max |mean over voxels| after subtraction:", float(np.abs(X.mean(axis=1)).max()))
print("X:", X.shape, X.dtype)

## 2.1 PCA and choice of the number of components

In [ ]:
n_fit = 60      
n_keep = 10     
t0 = time.time()
pca = PCA(n_components=n_fit, svd_solver="full")   
pca.fit(X)
print("PCA done in", round(time.time() - t0), "s")

ratios = pca.explained_variance_ratio_
cumulative_ratios = np.cumsum(ratios)
print(f"Variance explained by the {n_keep} kept components: {100 * cumulative_ratios[n_keep - 1]:.1f} %")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(1, n_fit + 1)

ax[0].plot(x, ratios, c='k', label='explained variance ratio')
ax[0].scatter([n_keep], [ratios[n_keep - 1]], s=150, marker='x', color='r', label=f'cutoff ({n_keep} components)')
ax[1].plot(x, cumulative_ratios, c='k', label='cumulative explained variance')
ax[1].hlines(y=cumulative_ratios[n_keep - 1], xmin=1, xmax=n_fit, linestyle='--', color='r',
             label=f'cutoff ({100 * cumulative_ratios[n_keep - 1]:.0f} %)')

for k in range(2):
    ax[k].set_xlabel('Component #', size=15)
    ax[k].legend(prop={'size': 15})
    ax[k].tick_params(axis='both', which='major', labelsize=15)
    ax[k].grid(True, alpha=0.3)
fig.savefig(op.join(FIG_DIR, "PCA_explained_variance.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
PCA_DIR = op.join(RESULTS_DIR, "PCA")        
os.makedirs(PCA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 2.2 Loading maps of the 10 kept components (axial view) 

In [ ]:
pca_clusters = []                                    
for k in range(n_keep):
    vol = np.zeros(vol_shape, dtype=np.float32)
    vol[slice_non_background] = pca.components_[k]
    pca_clusters.append(vol)
    nib.save(nib.Nifti1Image(vol, affine), op.join(PCA_DIR, f"pca_loading_{k + 1:02d}.nii.gz"))   
print(len(pca_clusters), "loading maps saved in", PCA_DIR)

In [ ]:
AXIAL_CUTS = (-20, -10, 0, 10, 20, 30, 40, 50)
for k, vol in enumerate(pca_clusters):
    thr = 2 * vol[slice_non_background].std()
    disp = plot_stat_map(nib.Nifti1Image(vol, affine), bg_img=ANAT_PATH, threshold=thr, display_mode="z",
                         cut_coords=AXIAL_CUTS, colorbar=True,
                         title=f"PC{k + 1} loading ({100 * ratios[k]:.1f} % of the variance, |loading| > 2 SD)")
    disp.savefig(op.join(FIG_DIR, f"PCA_loading_{k + 1:02d}.png"), dpi=150)
    plt.show()

In [ ]:
ncols = 2
nrows = int(np.ceil(n_keep / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 1.9 * nrows))
for k, ax in enumerate(np.atleast_1d(axes).ravel()):
    if k >= n_keep:
        ax.axis("off")
        continue
    thr = 2 * pca_clusters[k][slice_non_background].std()
    plot_stat_map(nib.Nifti1Image(pca_clusters[k], affine), bg_img=ANAT_PATH, threshold=thr, display_mode="z",
                  cut_coords=(-20, -5, 10, 25, 40, 55), colorbar=False, axes=ax,
                  title=f"PC{k + 1} ({100 * ratios[k]:.1f} %)")
fig.savefig(op.join(FIG_DIR, "PCA_loadings_overview.png"), dpi=200, bbox_inches="tight")
plt.show()

## 3. Correlation between the first 10 PC scores and the GLM regressors

In [ ]:
dm_path = op.join(RESULTS_DIR, "design_matrix.csv")
assert op.exists(dm_path), "Run 02_glm.ipynb first (it saves the design matrix)"
design_matrix = pd.read_csv(dm_path, index_col=0)
print("design matrix read from", dm_path)

reg = design_matrix.drop(columns=["constant"])
reg["story - math"] = reg["story"] - reg["math"]
first = ["story", "math", "story - math"]
reg = reg[first + [c for c in reg.columns if c not in first]]
print("regressors:", reg.columns.tolist())

In [ ]:
scores = pca.transform(X)                             
n_corr = 10
score_df = pd.DataFrame(scores[:, :n_corr], columns=[f"PC{i + 1}" for i in range(n_corr)])
pd.DataFrame(scores[:, :n_keep], columns=[f"PC{i + 1}" for i in range(n_keep)]).to_csv(op.join(PCA_DIR, "pca_scores.csv"), index=False)

S = scores[:, :n_corr]
R = reg.to_numpy(dtype=float)
corr = pd.DataFrame(
    [[np.corrcoef(S[:, i], R[:, j])[0, 1] for j in range(R.shape[1])] for i in range(n_corr)],
    index=score_df.columns, columns=reg.columns)
corr.round(3).to_csv(op.join(PCA_DIR, "pca_vs_glm_correlation.csv"))

fig, ax = plt.subplots(figsize=(13, 5.5))
im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(corr.shape[1])); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(corr.shape[0])); ax.set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
plt.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation between the scores of the first 10 principal components and the GLM regressors")
plt.tight_layout()
fig.savefig(op.join(FIG_DIR, "PCA_correlation_matrix.png"), dpi=200, bbox_inches="tight")
plt.show()